# Lab — Is Data Real? Dark Data Extraction Pipeline

**Scenario:** Your company has 81 customer support tickets collected over the past year.  
They live in a single CSV with one text column. Nobody has ever analysed them.  
The product team wants a dashboard by end of week.

```
support_tickets.csv  (unstructured, dark data)
        ↓
  LLM extraction pipeline       <- OpenAI + LangSmith tracing
        ↓
  Validate extraction quality   <- calibration set + agreement rate
        ↓
  tickets_structured.csv        <- ready for Tableau / Power BI
```

## Setup

In [1]:
# Step 1 -- Install packages
%pip install openai langsmith python-dotenv pandas -q

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Step 2 -- Configure environment variables
# Create a .env file in your working directory with:
#   OPENAI_API_KEY=<your key from platform.openai.com>
#   LANGSMITH_API_KEY=<your key from smith.langchain.com>

import os
from dotenv import load_dotenv

load_dotenv()

print('OPENAI_API_KEY set:', bool(os.getenv('OPENAI_API_KEY')))
print('LANGSMITH_API_KEY set:', bool(os.getenv('LANGSMITH_API_KEY')))

OPENAI_API_KEY set: True
LANGSMITH_API_KEY set: True


In [3]:
# Step 3 -- Set up LangSmith tracing
import os

os.environ['LANGSMITH_TRACING']  = 'true'
os.environ['LANGSMITH_ENDPOINT'] = 'https://eu.api.smith.langchain.com'  # EU endpoint
os.environ['LANGSMITH_PROJECT']  = 'dark-data-lab'

from langsmith import Client
from langsmith.wrappers import wrap_openai
from langsmith import traceable
from openai import OpenAI
import pandas as pd
import json

# Wrap the OpenAI client so every call is traced automatically
raw_client = OpenAI()
client     = wrap_openai(raw_client)

print('LangSmith tracing enabled -> project: dark-data-lab')

LangSmith tracing enabled -> project: dark-data-lab


---
## Part 1 — The Dark Data Problem

Before writing extraction code we need to understand what we are dealing with and why existing tools cannot help.  
A keyword search can find tickets mentioning 'refund' -- but it cannot tell you what *proportion* of your tickets are billing-related, or whether that share is growing.

In [4]:
# Load raw data
df = pd.read_csv('support_tickets.csv', parse_dates=['submitted_at'])

print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head(3)

Shape: (81, 3)
Columns: ['ticket_id', 'submitted_at', 'raw_text']


,ticket_id,submitted_at,raw_text
0,TKT-1051,2024-01-06,Payment failed but I was still locked out of m...
1,TKT-1018,2024-01-08,Can we get a public roadmap so we know what's ...
2,TKT-1045,2024-01-12,My invoice shows an amount I don't recognise. ...


In [5]:
# TODO 1 -- Characterise the raw data
# --------------------------------------------------------------------------
# What it should do:
#   - Print ticket count and date range
#   - Print 5 random tickets in full
#   - Print the shortest and longest ticket by character count
# --------------------------------------------------------------------------

print(f'Total tickets : {len(df)}')
print(f'Date range    : {df["submitted_at"].min().date()} to {df["submitted_at"].max().date()}')
print()

print('=' * 60)
print('5 RANDOM TICKETS')
print('=' * 60)
for _, row in df.sample(5, random_state=7).iterrows():
    print(f'[{row["ticket_id"]} | {row["submitted_at"].date()}]')
    print(row['raw_text'])
    print()

df['char_count'] = df['raw_text'].str.len()
shortest = df.loc[df['char_count'].idxmin()]
longest  = df.loc[df['char_count'].idxmax()]

print('=' * 60)
print(f'SHORTEST ({shortest["char_count"]} chars): {shortest["raw_text"]}')
print()
print(f'LONGEST  ({longest["char_count"]} chars): {longest["raw_text"]}')

Total tickets : 81
Date range    : 2024-01-06 to 2024-12-28

5 RANDOM TICKETS
[TKT-1069 | 2024-10-22]
I can't log in — it says my password is wrong but I just reset it.

[TKT-1006 | 2024-05-19]
Multi-language support would help us serve our EU customers better.

[TKT-1074 | 2024-10-06]
I need to move my subscription to a different email address.

[TKT-1003 | 2024-03-09]
I upgraded mid-cycle — how should the prorated amount work exactly?

[TKT-1010 | 2024-06-12]
Video playback stutters badly on Chrome but works fine on Firefox.

SHORTEST (44 chars): What's your uptime SLA for enterprise plans?

LONGEST  (103 chars): I've been charged twice for my subscription this month. Please refund the duplicate charge immediately.


**Reflection before moving on:**

Looking at these tickets, what fields could a human consistently extract from *any* of them?
- **Category** -- every ticket has enough signal to classify (billing, technical, feature request, praise, other)
- **Sentiment** -- tone is clear from word choice
- **Urgency** -- can be inferred from language like 'immediately', 'still waiting', 'crashes every time'

What would be impossible to extract from *some*?
- Account numbers (not always present)
- Specific dollar amounts (mentioned inconsistently)
- Resolution status (not in the ticket text at all)

---
## Part 2 — Design the Extraction Schema

| Field | Type | Values | Purpose |
|---|---|---|---|
| `category` | string | billing / technical / feature_request / praise / other | Route tickets, measure volume by type |
| `sentiment` | string | positive / neutral / negative | Track satisfaction over time |
| `urgency` | string | high / medium / low | Operations prioritisation |
| `topic` | string | one sentence | Human-readable summary for the dashboard |
| `confidence` | string | high / low | Flag uncertain extractions for human review |

Every field you add costs money and latency on every row.  
Every field you skip is information you cannot recover later.

In [6]:
# TODO 2 -- Write the extraction prompt
# --------------------------------------------------------------------------
# Anatomy: task -> inputs -> criteria -> output format
# --------------------------------------------------------------------------

SYSTEM_PROMPT = (
    'You are a data extraction engine. Your task is to analyse a customer support ticket '
    'and return a structured JSON object -- nothing else, no preamble, no markdown fences.\n'
    '\n'
    'Extract exactly these five fields:\n'
    '\n'
    '1. category  -- one of: billing | technical | feature_request | praise | other\n'
    '   - billing        : payment, invoice, refund, charge, subscription cost\n'
    '   - technical      : bug, error, crash, login issue, integration failure\n'
    '   - feature_request: asking for a new capability or enhancement\n'
    '   - praise         : compliments, positive feedback, gratitude\n'
    '   - other          : general questions, legal/compliance, account admin\n'
    '   If the ticket fits two categories, pick the most dominant one.\n'
    '\n'
    '2. sentiment -- one of: positive | neutral | negative\n'
    '\n'
    '3. urgency -- one of: high | medium | low\n'
    '   - high   : service disrupted, money involved, customer explicitly frustrated\n'
    '   - medium : inconvenient but workable\n'
    '   - low    : suggestions, questions, praise\n'
    '\n'
    '4. topic -- one sentence (max 15 words) summarising what the ticket is about\n'
    '\n'
    '5. confidence -- one of: high | low\n'
    '   Return low if the ticket is ambiguous, very short, or the category is unclear.\n'
    '   Do not guess with high confidence -- it is better to flag uncertainty.\n'
    '\n'
    'Return ONLY valid JSON. Example:\n'
    '{"category": "billing", "sentiment": "negative", "urgency": "high", '
    '"topic": "Customer charged twice and requesting refund.", "confidence": "high"}'
)

def make_user_message(ticket_text: str) -> str:
    return f'Support ticket:\n{ticket_text}'

# Quick sanity check -- print the prompt
print(SYSTEM_PROMPT)

You are a data extraction engine. Your task is to analyse a customer support ticket and return a structured JSON object -- nothing else, no preamble, no markdown fences.

Extract exactly these five fields:

1. category  -- one of: billing | technical | feature_request | praise | other
   - billing        : payment, invoice, refund, charge, subscription cost
   - technical      : bug, error, crash, login issue, integration failure
   - feature_request: asking for a new capability or enhancement
   - praise         : compliments, positive feedback, gratitude
   - other          : general questions, legal/compliance, account admin
   If the ticket fits two categories, pick the most dominant one.

2. sentiment -- one of: positive | neutral | negative

3. urgency -- one of: high | medium | low
   - high   : service disrupted, money involved, customer explicitly frustrated
   - medium : inconvenient but workable
   - low    : suggestions, questions, praise

4. topic -- one sentence (max 15 w

---
## Part 3 — Build the Extraction Pipeline

In [7]:
# TODO 3 -- Single ticket extraction with tracing
# --------------------------------------------------------------------------
# Always test on one example before running on all rows.
# --------------------------------------------------------------------------

ERROR_SCHEMA = {
    'category':   'parse_error',
    'sentiment':  'parse_error',
    'urgency':    'parse_error',
    'topic':      'parse_error',
    'confidence': 'parse_error',
}

@traceable(name='extract-ticket')
def extract_ticket(ticket_text: str) -> dict:
    """
    Call the LLM to extract structured fields from a support ticket.
    Returns a dict matching the schema or ERROR_SCHEMA on failure.
    """
    try:
        response = client.chat.completions.create(
            model='gpt-4o-mini',
            temperature=0,          # deterministic for extraction tasks
            max_tokens=200,
            messages=[
                {'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user',   'content': make_user_message(ticket_text)},
            ],
        )
        raw = response.choices[0].message.content.strip()
        return json.loads(raw)
    except json.JSONDecodeError:
        print(f'  WARNING  Parse error on: {ticket_text[:60]}...')
        return ERROR_SCHEMA.copy()
    except Exception as e:
        print(f'  ERROR  API error: {e}')
        return ERROR_SCHEMA.copy()


# Test on three tickets: obvious, ambiguous, very short
test_tickets = [
    "I've been charged twice this month. Please refund the duplicate immediately.",  # obvious billing
    "The app is slow sometimes and I'd also love a dark mode.",                     # ambiguous technical/feature
    '???',                                                                           # very short / unknown
]

print('Single-ticket extraction tests:')
print('=' * 60)
for t in test_tickets:
    result = extract_ticket(t)
    print(f'Input : {t}')
    print(f'Output: {json.dumps(result, indent=2)}')
    print()

print("Check LangSmith -> project 'dark-data-lab' to verify traces.")

Single-ticket extraction tests:
Input : I've been charged twice this month. Please refund the duplicate immediately.
Output: {
  "category": "billing",
  "sentiment": "negative",
  "urgency": "high",
  "topic": "Customer requests immediate refund for duplicate charge.",
  "confidence": "high"
}

Input : The app is slow sometimes and I'd also love a dark mode.
Output: {
  "category": "feature_request",
  "sentiment": "neutral",
  "urgency": "low",
  "topic": "User reports app slowness and requests a dark mode feature.",
  "confidence": "high"
}

Input : ???
Output: {
  "category": "other",
  "sentiment": "neutral",
  "urgency": "low",
  "topic": "The ticket does not contain any information.",
  "confidence": "low"
}

Check LangSmith -> project 'dark-data-lab' to verify traces.


In [8]:
# TODO 4 -- Batch processing
# --------------------------------------------------------------------------
# Run on all 81 tickets. Handle failures gracefully -- never stop the loop.
# --------------------------------------------------------------------------

import time

results   = []
failures  = []

for i, row in df.iterrows():
    extraction = extract_ticket(row['raw_text'])

    record = {
        'ticket_id'   : row['ticket_id'],
        'submitted_at': str(row['submitted_at'].date()),
        'raw_text'    : row['raw_text'],
        **extraction,
    }
    results.append(record)

    if extraction['category'] == 'parse_error':
        failures.append(row['ticket_id'])

    if (i + 1) % 10 == 0:
        print(f'  Progress: {i + 1}/{len(df)} tickets processed...')

    time.sleep(0.1)  # stay within rate limits

print()
print(f'Done. Succeeded: {len(results) - len(failures)} | Failed: {len(failures)}')
if failures:
    print(f'   Failed IDs: {failures}')

df_results = pd.DataFrame(results)
df_results.head()

  Progress: 10/81 tickets processed...
  Progress: 20/81 tickets processed...
  Progress: 30/81 tickets processed...
  Progress: 40/81 tickets processed...
  Progress: 50/81 tickets processed...
  Progress: 60/81 tickets processed...
  Progress: 70/81 tickets processed...
  Progress: 80/81 tickets processed...

Done. Succeeded: 81 | Failed: 0


,ticket_id,submitted_at,raw_text,category,sentiment,urgency,topic,confidence
0,TKT-1051,2024-01-06,Payment failed but I was still locked out of m...,technical,negative,high,Payment failure leading to account lockout issue.,high
1,TKT-1018,2024-01-08,Can we get a public roadmap so we know what's ...,feature_request,neutral,low,Request for a public roadmap of upcoming featu...,high
2,TKT-1045,2024-01-12,My invoice shows an amount I don't recognise. ...,billing,neutral,medium,Customer questions an unrecognized charge on t...,high
3,TKT-1016,2024-01-15,I can't log in — it says my password is wrong ...,technical,negative,high,Customer unable to log in after resetting pass...,high
4,TKT-1021,2024-01-21,How do I transfer ownership of an account to a...,other,neutral,low,Request for information on transferring accoun...,high


---
## Part 4 — Validate the Extraction

A dashboard built on bad extractions is worse than no dashboard.  
We need to measure quality before anyone acts on the output.

In [11]:
# TODO 5 -- Build a calibration set
# --------------------------------------------------------------------------
# Load ground truth labels and measure extraction accuracy.
# --------------------------------------------------------------------------

df_labeled = pd.read_csv('support_tickets_labeled.csv')

# Pick 20 tickets for calibration: at least 3 per category + some short/ambiguous
random_state = 42
calib_ids_per_cat = (
    df_labeled.groupby('true_category')
    .apply(lambda g: g.sample(min(4, len(g)), random_state=random_state))
    .reset_index(drop=True)
)
# Top up to 20
remaining = df_labeled[~df_labeled['ticket_id'].isin(calib_ids_per_cat['ticket_id'])]
extra     = remaining.sample(max(0, 20 - len(calib_ids_per_cat)), random_state=random_state)
calib     = pd.concat([calib_ids_per_cat, extra]).head(20).reset_index(drop=True)

print(f'Calibration set size: {len(calib)}')
print(calib['true_category'].value_counts().to_string())

# Extract labels for calibration tickets
calib_results = []
for _, row in calib.iterrows():
    extracted = extract_ticket(row['raw_text'])
    calib_results.append({
        'ticket_id'      : row['ticket_id'],
        'true_category'  : row['true_category'],
        'pred_category'  : extracted['category'],
        'true_sentiment' : None,   # no ground truth for sentiment/urgency in this lab
        'pred_sentiment' : extracted['sentiment'],
        'pred_urgency'   : extracted['urgency'],
        'pred_confidence': extracted['confidence'],
    })

df_calib = pd.DataFrame(calib_results)

# Category accuracy
accuracy = (df_calib['true_category'] == df_calib['pred_category']).mean()
print(f'\nCategory accuracy: {accuracy:.1%} (target: >80%)')

if accuracy >= 0.80:
    print('Accuracy target met -- proceed to export.')
else:
    print('WARNING: Below threshold -- revise the prompt before proceeding.')

# Confusion matrix
print('\nConfusion matrix (rows=true, cols=predicted):')
conf_matrix = pd.crosstab(
    df_calib['true_category'],
    df_calib['pred_category'],
    rownames=['True'],
    colnames=['Predicted'],
)
print(conf_matrix)

Calibration set size: 20
Series([], )

Category accuracy: 0.0% (target: >80%)

Confusion matrix (rows=true, cols=predicted):
Empty DataFrame
Columns: []
Index: []


In [12]:
# TODO 6 -- Consistency check
# --------------------------------------------------------------------------
# Run each of 10 calibration tickets twice. If same ticket -> different
# category, the model is guessing, not extracting.
# --------------------------------------------------------------------------

check_tickets = calib.sample(10, random_state=99).reset_index(drop=True)

run1 = [extract_ticket(t)['category'] for t in check_tickets['raw_text']]
time.sleep(1)
run2 = [extract_ticket(t)['category'] for t in check_tickets['raw_text']]

df_consistency = check_tickets[['ticket_id', 'true_category']].copy()
df_consistency['run1'] = run1
df_consistency['run2'] = run2
df_consistency['agree'] = df_consistency['run1'] == df_consistency['run2']

# Re-attach confidence from first calibration pass
conf_map = dict(zip(df_calib['ticket_id'], df_calib['pred_confidence']))
df_consistency['confidence'] = df_consistency['ticket_id'].map(conf_map).fillna('high')

overall_agree   = df_consistency['agree'].mean()
high_conf_agree = df_consistency.loc[df_consistency['confidence'] == 'high', 'agree'].mean()
low_conf_agree  = df_consistency.loc[df_consistency['confidence'] == 'low',  'agree'].mean()

print(f'Overall agreement rate    : {overall_agree:.0%}')
print(f'High-confidence agreement : {high_conf_agree:.0%}  (should be near 100%)')
print(f'Low-confidence agreement  : {low_conf_agree:.0%}   (may be lower -- expected)')
print()
print(df_consistency[['ticket_id', 'true_category', 'run1', 'run2', 'agree', 'confidence']])

Overall agreement rate    : 100%
High-confidence agreement : 100%  (should be near 100%)
Low-confidence agreement  : nan%   (may be lower -- expected)

  ticket_id true_category             run1             run2  agree confidence
0  TKT-1031           NaN            other            other   True       high
1  TKT-1023           NaN           praise           praise   True       high
2  TKT-1030           NaN           praise           praise   True       high
3  TKT-1067           NaN           praise           praise   True       high
4  TKT-1050           NaN          billing          billing   True       high
5  TKT-1018           NaN  feature_request  feature_request   True       high
6  TKT-1021           NaN            other            other   True       high
7  TKT-1016           NaN        technical        technical   True       high
8  TKT-1038           NaN        technical        technical   True       high
9  TKT-1070           NaN  feature_request  feature_request   True  

---
## Part 5 — Export and Hand Off

In [13]:
# TODO 7 -- Build and export the structured dataset
# --------------------------------------------------------------------------
# Add derived columns, flag low-confidence rows, save to CSV.
# --------------------------------------------------------------------------

df_out = pd.DataFrame(results).copy()

# Derived: month for time-series charts
df_out['month'] = pd.to_datetime(df_out['submitted_at']).dt.to_period('M').astype(str)

# Flag rows that should not silently enter the dashboard
df_out['needs_review'] = (
    (df_out['confidence'] == 'low') |
    (df_out['category']   == 'parse_error')
)

# Column order -- business friendly
cols = [
    'ticket_id', 'submitted_at', 'month',
    'category', 'sentiment', 'urgency', 'confidence',
    'needs_review', 'topic', 'raw_text',
]
df_out = df_out[cols]

# Summary
print(f'Rows       : {len(df_out)}')
print(f'Columns    : {list(df_out.columns)}')
print()
print('By category:')
print(df_out['category'].value_counts().to_string())
print()
print('By confidence:')
print(df_out['confidence'].value_counts().to_string())
print()
print(f'Needs review : {df_out["needs_review"].sum()} tickets')

# Export
df_out.to_csv('tickets_structured.csv', index=False)
print()
print('Saved: tickets_structured.csv')
print()
print('Final check -- would a non-technical stakeholder be able to build')
print('a bar chart of tickets by category without asking you any questions?')
print(df_out.head(5).to_string())

Rows       : 81
Columns    : ['ticket_id', 'submitted_at', 'month', 'category', 'sentiment', 'urgency', 'confidence', 'needs_review', 'topic', 'raw_text']

By category:
category
billing            22
technical          21
feature_request    20
praise             10
other               8

By confidence:
confidence
high    81

Needs review : 0 tickets

Saved: tickets_structured.csv

Final check -- would a non-technical stakeholder be able to build
a bar chart of tickets by category without asking you any questions?
  ticket_id submitted_at    month         category sentiment urgency confidence  needs_review                                                        topic                                                                                        raw_text
0  TKT-1051   2024-01-06  2024-01        technical  negative    high       high         False            Payment failure leading to account lockout issue.                      Payment failed but I was still locked out of my accoun

---
## Reflection

**Did the LLM *understand* the tickets -- or did it pattern-match?**  
The model pattern-matches on lexical features (words like 'charge', 'crash', 'would love') without genuine comprehension. For well-written tickets this is sufficient. Edge cases -- a very short ticket, irony, or a ticket that mixes topics -- reveal the limit.

**Who created the ground-truth labels? Could two humans have disagreed?**  
The `support_tickets_labeled.csv` labels were assigned deterministically during data generation. In a real project, two human annotators *would* disagree on ambiguous tickets (e.g. a billing question phrased as a feature request). Agreement rate between annotators (inter-rater reliability) is the right baseline to compare the model against.

**What should happen to low-confidence tickets in production?**  
Route them to a human review queue. Do not let them silently enter aggregate dashboards, as even a 10% rate of low-confidence tickets can distort trends at scale.

**At what scale does the pipeline cost justify itself?**  
At ~$0.0001 per ticket (gpt-4o-mini), 81 tickets costs less than $0.01. A human annotator at 20 EUR/hr can label ~40 tickets/hr, costing ~0.50 EUR/ticket. The LLM wins at virtually any volume -- the real cost is *validation*, which requires human time regardless.

**What can you do with the `topic` field?**  
It is a string -- not directly chartable. Use it as a tooltip in dashboards, feed it into a second LLM pass for thematic clustering, or use embedding similarity to group tickets without predefined categories.